# 🧠 ML LeetCode Interview Guide: Data Structures & Algorithms in Pure Python
### Core ML, Probability, and Systems Problems Implemented from Scratch

This notebook covers **LeetCode-style algorithmic questions** tailored for Machine Learning Engineer (MLE) and Applied Scientist interviews. Unlike classical model implementation rounds, these questions evaluate **algorithmic thinking, data structure selection, numerical edge cases, and time/space complexity** in **pure Python** (using only standard libraries: `math`, `random`, `bisect`, `heapq`, `collections`).

---

## 📌 Table of Contents
1. [Weighted Random Sampling (Prefix Sums & Binary Search)](#1-weighted-random-sampling)
2. [Reservoir Sampling (Uniform Streaming with $\mathcal{O}(1)$ Memory)](#2-reservoir-sampling)
3. [Sort Transformed Array (Convex Quadratic & Two Pointers)](#3-sort-transformed-array)
4. [Find Minimum of a Convex Function (Ternary Search)](#4-find-minimum-of-a-convex-function)
5. [Expected Calibration Error - ECE (Bucket Binning)](#5-expected-calibration-error-ece)
6. [Exact ROC-AUC with Tie Handling (Wilcoxon Rank-Sum)](#6-exact-roc-auc-with-tie-handling)
7. [Precision-Recall Curve & Average Precision (Sweep-Line)](#7-precision-recall-curve--average-precision)
8. [Dot Product of Two Sparse Vectors (Two Pointers & Binary Search)](#8-dot-product-of-two-sparse-vectors)
9. [Top-K Cosine Similarity Retrieval (Min-Heap with `heapq`)](#9-top-k-cosine-similarity-retrieval)
10. [TF-IDF & Inverted Index Document Retrieval](#10-tf-idf--inverted-index-document-retrieval)
11. [Streaming Running Median (Two Heaps)](#11-streaming-running-median)
12. [Edit Distance & Word Error Rate - WER (Dynamic Programming)](#12-edit-distance--word-error-rate-wer)
13. [Image Rotation by Arbitrary Angle (Inverse Mapping & Bilinear Interpolation)](#13-image-rotation-by-arbitrary-angle)


---
## 1. Weighted Random Sampling
**LeetCode Analog**: [LC 528: Random Pick with Weight](https://leetcode.com/problems/random-pick-with-weight/)  
**ML Context**: Sampling tokens with temperature in LLMs, categorical distribution sampling, AdaBoost sample reweighting.

### Algorithmic Principle
Given positive weights $w = [w_0, w_1, \dots, w_{N-1}]$:
1. Construct cumulative prefix sums: $S_i = \sum_{j=0}^i w_j$.
2. Draw a uniform random float $u \in [0, S_{N-1})$.
3. Use binary search (`bisect.bisect_right`) to find the smallest index $i$ where $u < S_i$.

- **Time Complexity**: $\mathcal{O}(N)$ initialization, $\mathcal{O}(\log N)$ per sample.
- **Space Complexity**: $\mathcal{O}(N)$ for prefix sums.
- **Interview Trap**: Using linear scan $\mathcal{O}(N)$ instead of binary search $\mathcal{O}(\log N)$; handling floating-point roundoff at boundary $S_{N-1}$.


In [ ]:
import random
import bisect
from collections import Counter
from typing import List

class WeightedRandomSampler:
    """
    Weighted random sampling using Prefix Sums + Binary Search.
    Time: O(N) init, O(log N) sample. Space: O(N).
    """
    def __init__(self, weights: List[float]):
        if not weights or any(w < 0 for w in weights):
            raise ValueError("Weights must be non-empty and non-negative.")
        
        self.prefix_sums: List[float] = []
        running_sum = 0.0
        for w in weights:
            running_sum += w
            self.prefix_sums.append(running_sum)
        self.total_weight = running_sum

    def sample(self) -> int:
        target = random.random() * self.total_weight
        # bisect_right finds the first prefix_sum strictly greater than target
        return bisect.bisect_right(self.prefix_sums, target)

# Verification Test
random.seed(42)
weights = [1.0, 3.0, 6.0]  # Expected: ~10%, ~30%, ~60%
sampler = WeightedRandomSampler(weights)
trials = 10000
samples = [sampler.sample() for _ in range(trials)]
counts = Counter(samples)

print("Empirical distribution over 10,000 samples:")
for idx, weight in enumerate(weights):
    prob = counts[idx] / trials
    expected = weight / sum(weights)
    print(f"  Index {idx}: observed = {prob:.4f}, expected = {expected:.4f}")


---
## 2. Reservoir Sampling
**LeetCode Analog**: [LC 382: Linked List Random Node](https://leetcode.com/problems/linked-list-random-node/), [LC 398: Random Pick Index](https://leetcode.com/problems/random-pick-index/)  
**ML Context**: Sampling from infinite/unknown streaming data pipelines, replay buffers in reinforcement learning without loading full stream into memory.

### Algorithmic Principle
To sample $k$ items uniformly from a stream of unknown length $N$:
1. Store the first $k$ elements into the reservoir.
2. For each incoming $i$-th element ($i \ge k$, 0-indexed):
   - Pick a random integer $j \in [0, i]$.
   - If $j < k$, replace `reservoir[j]` with the new element.
3. Every element seen so far has an exact $\frac{k}{i+1}$ probability of being in the reservoir.

- **Time Complexity**: $\mathcal{O}(N)$ stream scan.
- **Space Complexity**: $\mathcal{O}(k)$ auxiliary memory.
- **Interview Trap**: Random range off-by-one errors (must pick $j \in [0, i]$ inclusive).


In [ ]:
from typing import List, Iterable, Any
import random
from collections import Counter

def reservoir_sample(stream: Iterable[Any], k: int) -> List[Any]:
    """
    Uniformly samples k items from an incoming stream of unknown length.
    Time: O(N), Space: O(k).
    """
    if k <= 0:
        return []
    
    reservoir = []
    for i, item in enumerate(stream):
        if i < k:
            reservoir.append(item)
        else:
            j = random.randint(0, i)
            if j < k:
                reservoir[j] = item
    return reservoir

# Verification Test: Monte Carlo uniformity check
random.seed(42)
stream_items = list(range(10))  # Stream items: 0..9
k = 2
trials = 10000
freq = Counter()

for _ in range(trials):
    sample = reservoir_sample(stream_items, k)
    freq.update(sample)

print(f"Sample frequency over {trials} trials (expected ~{trials * (k / 10):.0f} per item):")
for item in sorted(freq.keys()):
    print(f"  Item {item}: count = {freq[item]}")


---
## 3. Sort Transformed Array under Convex Quadratic Function
**LeetCode Analog**: [LC 360: Sort Transformed Array](https://leetcode.com/problems/sort-transformed-array/)  
**ML Context**: Evaluating convex/concave quadratic losses ($f(x) = ax^2 + bx + c$) over sorted inputs; exploiting convexity to avoid $\mathcal{O}(N \log N)$ sorting.

### Algorithmic Principle
A quadratic function $f(x) = ax^2 + bx + c$ has a single extremum (vertex at $x = -b / (2a)$):
- **Convex ($a > 0$)**: Parabola opens upward; maximums occur at the outer boundaries. Use two pointers (`left=0`, `right=N-1`) and fill the result array **from back to front**.
- **Concave ($a < 0$)**: Parabola opens downward; minimums occur at the outer boundaries. Fill the result array **from front to back**.
- **Linear ($a = 0$)**: If $b \ge 0$, array is already sorted; if $b < 0$, reverse array.

- **Time Complexity**: $\mathcal{O}(N)$ linear time.
- **Space Complexity**: $\mathcal{O}(N)$ for output array.
- **Interview Trap**: Forgetting to handle $a < 0$ vs $a > 0$ pointer fill directions.


In [ ]:
from typing import List

def sort_transformed_array(nums: List[int], a: int, b: int, c: int) -> List[int]:
    """
    Transforms sorted nums by f(x) = a*x^2 + b*x + c and returns sorted list in O(N).
    Time: O(N), Space: O(N).
    """
    def f(x: int) -> int:
        return a * x * x + b * x + c

    n = len(nums)
    res = [0] * n
    left, right = 0, n - 1

    if a >= 0:
        # Convex: fill largest values from back to front
        idx = n - 1
        while left <= right:
            fl, fr = f(nums[left]), f(nums[right])
            if fl >= fr:
                res[idx] = fl
                left += 1
            else:
                res[idx] = fr
                right -= 1
            idx -= 1
    else:
        # Concave: fill smallest values from front to back
        idx = 0
        while left <= right:
            fl, fr = f(nums[left]), f(nums[right])
            if fl <= fr:
                res[idx] = fl
                left += 1
            else:
                res[idx] = fr
                right -= 1
            idx += 1

    return res

# Verification Test
nums = [-4, -2, 2, 4]
# Convex case (a = 1 > 0)
res_convex = sort_transformed_array(nums, a=1, b=3, c=5)
print("Convex transformed (a=1, b=3, c=5):", res_convex)
assert res_convex == sorted([1*x**2 + 3*x + 5 for x in nums])

# Concave case (a = -1 < 0)
res_concave = sort_transformed_array(nums, a=-1, b=3, c=5)
print("Concave transformed (a=-1, b=3, c=5):", res_concave)
assert res_concave == sorted([-1*x**2 + 3*x + 5 for x in nums])


---
## 4. Find Minimum of a Convex Function (Ternary Search)
**LeetCode Analog**: [LC 852: Peak Index in a Mountain Array](https://leetcode.com/problems/peak-index-in-a-mountain-array/) (Continuous variant)  
**ML Context**: Derivative-free 1D line search, tuning continuous hyperparameters with unimodal validation loss (e.g. learning rate, regularization weight).

### Algorithmic Principle
For a strictly unimodal function $f(x)$ on $[a, b]$:
1. Divide $[a, b]$ into three equal segments using two midpoints:
   $$m_1 = a + \frac{b - a}{3}, \quad m_2 = b - \frac{b - a}{3}$$
2. If $f(m_1) < f(m_2)$, the minimum cannot lie in $(m_2, b]$; set $b = m_2$.
3. Otherwise, the minimum cannot lie in $[a, m_1)$; set $a = m_1$.
4. Repeat until $|b - a| < \epsilon$.

- **Time Complexity**: $\mathcal{O}(\log((b - a)/\epsilon))$.
- **Space Complexity**: $\mathcal{O}(1)$.
- **Interview Trap**: Confusing strict unimodality with local plateau regions where ternary search can get stuck.


In [ ]:
from typing import Callable

def ternary_search_min(f: Callable[[float], float], a: float, b: float, tol: float = 1e-7) -> float:
    """
    Finds x that minimizes unimodal function f on [a, b].
    Time: O(log((b - a) / tol)), Space: O(1).
    """
    left, right = a, b
    while right - left > tol:
        m1 = left + (right - left) / 3.0
        m2 = right - (right - left) / 3.0
        if f(m1) < f(m2):
            right = m2
        else:
            left = m1
    return (left + right) / 2.0

# Verification Test: Minimize f(x) = (x - 3.5)^2 + 10 (minimum at x = 3.5)
f = lambda x: (x - 3.5)**2 + 10.0
x_min = ternary_search_min(f, a=0.0, b=10.0)

print(f"Optimal x found: {x_min:.6f} (True minimum = 3.500000)")
print(f"Function value at minimum: {f(x_min):.6f}")
assert abs(x_min - 3.5) < 1e-5


---
## 5. Expected Calibration Error (ECE)
**ML Context**: Model calibration. A well-calibrated classifier satisfies $\mathbb{P}(\hat{y} = y \mid \hat{p} = p) = p$. Required for confidence estimation, Platt scaling, and temperature scaling.

### Algorithmic Principle
1. Group $N$ predictions into $M$ equal-width bins $B_1, \dots, B_M$ over $[0, 1]$.
2. For each bin $B_m$ with $|B_m| > 0$:
   $$\text{acc}(B_m) = \frac{1}{|B_m|} \sum_{i \in B_m} \mathbb{I}(y_i = 1), \quad \text{conf}(B_m) = \frac{1}{|B_m|} \sum_{i \in B_m} p_i$$
3. Compute weighted calibration metrics:
   $$\text{ECE} = \sum_{m=1}^M \frac{|B_m|}{N} |\text{acc}(B_m) - \text{conf}(B_m)|, \quad \text{MCE} = \max_{m} |\text{acc}(B_m) - \text{conf}(B_m)|$$

- **Time Complexity**: $\mathcal{O}(N + M)$ single pass binning.
- **Space Complexity**: $\mathcal{O}(M)$ bin accumulators.
- **Interview Trap**: Edge case $p_i = 1.0$ mapping to index $M$ (clamp to $M - 1$).


In [ ]:
from typing import List, Dict

def compute_ece(probs: List[float], labels: List[int], n_bins: int = 10) -> Dict[str, float]:
    """
    Computes Expected Calibration Error (ECE) and Maximum Calibration Error (MCE).
    Time: O(N + M), Space: O(M).
    """
    if len(probs) != len(labels):
        raise ValueError("probs and labels must have the same length.")
    
    bin_counts = [0] * n_bins
    bin_correct = [0] * n_bins
    bin_conf_sum = [0.0] * n_bins

    for p, y in zip(probs, labels):
        # Map probability to bin [0, n_bins-1] with boundary clamping
        b_idx = min(int(p * n_bins), n_bins - 1)
        bin_counts[b_idx] += 1
        bin_correct[b_idx] += (1 if y == 1 else 0)
        bin_conf_sum[b_idx] += p

    total_samples = len(probs)
    ece = 0.0
    mce = 0.0

    print("Bin-level statistics:")
    for i in range(n_bins):
        if bin_counts[i] > 0:
            acc = bin_correct[i] / bin_counts[i]
            conf = bin_conf_sum[i] / bin_counts[i]
            gap = abs(acc - conf)
            ece += (bin_counts[i] / total_samples) * gap
            mce = max(mce, gap)
            bin_range = f"[{i/n_bins:.1f}, {(i+1)/n_bins:.1f})"
            print(f"  Bin {bin_range}: Count={bin_counts[i]}, Acc={acc:.3f}, Conf={conf:.3f}, Gap={gap:.3f}")

    return {"ece": ece, "mce": mce}

# Verification Test
probs = [0.1, 0.15, 0.2, 0.75, 0.8, 0.95]
labels = [0, 0, 1, 1, 1, 1]
res = compute_ece(probs, labels, n_bins=5)
print(f"\nECE: {res['ece']:.4f}, MCE: {res['mce']:.4f}")


---
## 6. Exact ROC-AUC with Tie Handling
**ML Context**: Ranking and classification evaluation. Measures $P(\hat{p}_+ > \hat{p}_-)$ — probability that a randomly chosen positive scores higher than a randomly chosen negative.

### Algorithmic Principle
Using the Wilcoxon-Mann-Whitney rank-sum formulation:
1. Sort pairs $(p_i, y_i)$ ascending by score.
2. Group tied prediction scores and assign them their **average rank**.
3. Compute sum of positive ranks $R_+ = \sum_{i \in \text{pos}} \text{rank}_i$.
4. Calculate AUC directly:
   $$\text{AUC} = \frac{R_+ - \frac{n_+ (n_+ + 1)}{2}}{n_+ \cdot n_-}$$

- **Time Complexity**: $\mathcal{O}(N \log N)$ to sort, $\mathcal{O}(N)$ rank sweep.
- **Space Complexity**: $\mathcal{O}(N)$.
- **Interview Trap**: Naive rank assignment inflates/deflates AUC when multiple candidates have identical prediction scores.


In [ ]:
from typing import List

def compute_roc_auc(probs: List[float], labels: List[int]) -> float:
    """
    Computes exact ROC-AUC with Wilcoxon rank-sum and average-rank tie handling.
    Time: O(N log N), Space: O(N).
    """
    if len(probs) != len(labels):
        raise ValueError("Length mismatch between probs and labels.")
    
    n_pos = sum(labels)
    n_neg = len(labels) - n_pos
    if n_pos == 0 or n_neg == 0:
        raise ValueError("ROC-AUC requires both positive and negative classes.")

    # Sort ascending by predicted score
    indexed = sorted(enumerate(zip(probs, labels)), key=lambda x: x[1][0])
    
    rank_sum_pos = 0.0
    i = 0
    n = len(indexed)
    
    # Sweep through items, handling tied intervals [i, j)
    while i < n:
        j = i
        while j < n and indexed[j][1][0] == indexed[i][1][0]:
            j += 1
        
        # 1-based ranks from (i + 1) to j have average rank:
        avg_rank = ((i + 1) + j) / 2.0
        for k in range(i, j):
            if indexed[k][1][1] == 1:
                rank_sum_pos += avg_rank
        i = j

    auc = (rank_sum_pos - (n_pos * (n_pos + 1) / 2.0)) / (n_pos * n_neg)
    return auc

# Verification Test
# 1. Distinct scores: positives at 0.35, 0.8; negatives at 0.1, 0.4 -> pairs: 3/4 = 0.75
probs = [0.1, 0.4, 0.35, 0.8]
labels = [0, 0, 1, 1]
print("AUC (distinct scores):", compute_roc_auc(probs, labels))
assert compute_roc_auc(probs, labels) == 0.75

# 2. Tied scores test: random guessing with identical predictions must yield exactly 0.5
probs_tied = [0.5, 0.5, 0.5, 0.5]
labels_tied = [1, 0, 1, 0]
print("AUC (tied scores):", compute_roc_auc(probs_tied, labels_tied))
assert compute_roc_auc(probs_tied, labels_tied) == 0.5


---
## 7. Precision-Recall Curve & Average Precision (AP)
**ML Context**: Information retrieval, highly imbalanced classification, object detection benchmark (mAP).

### Algorithmic Principle
1. Sort pairs $(p_i, y_i)$ descending by predicted probability.
2. Sweep down the list, updating running counts of $TP$ and $FP$.
3. At each step $k$, compute:
   $$\text{Recall}_k = \frac{TP_k}{\text{total\_pos}}, \quad \text{Precision}_k = \frac{TP_k}{TP_k + FP_k}$$
4. Integrate area under the curve via step-wise Riemann sum:
   $$\text{AP} = \sum_k (\text{Recall}_k - \text{Recall}_{k-1}) \cdot \text{Precision}_k$$

- **Time Complexity**: $\mathcal{O}(N \log N)$.
- **Space Complexity**: $\mathcal{O}(N)$.
- **Interview Trap**: Dividing by zero when no positives are predicted at early thresholds ($TP + FP = 0$).


In [ ]:
from typing import List

def compute_pr_auc(probs: List[float], labels: List[int]) -> float:
    """
    Computes Average Precision (Area under Precision-Recall curve).
    Time: O(N log N), Space: O(N).
    """
    total_pos = sum(labels)
    if total_pos == 0:
        return 0.0
    
    # Sort descending by predicted probability
    pairs = sorted(zip(probs, labels), key=lambda x: x[0], reverse=True)
    
    tp = 0
    fp = 0
    prev_recall = 0.0
    ap = 0.0

    print("PR curve sweep steps:")
    for p, label in pairs:
        if label == 1:
            tp += 1
        else:
            fp += 1
        
        recall = tp / total_pos
        precision = tp / (tp + fp)
        # Step area contribution: delta_recall * precision
        delta_recall = recall - prev_recall
        ap += delta_recall * precision
        prev_recall = recall
        print(f"  Score={p:.2f}, Label={label} -> Precision={precision:.3f}, Recall={recall:.3f}, delta_R={delta_recall:.3f}")

    return ap

# Verification Test
probs = [0.95, 0.85, 0.60, 0.20]
labels = [1, 1, 0, 1]
ap = compute_pr_auc(probs, labels)
print(f"\nAverage Precision (PR-AUC): {ap:.4f}")


---
## 8. Dot Product of Two Sparse Vectors
**LeetCode Analog**: [LC 1570: Dot Product of Two Sparse Vectors](https://leetcode.com/problems/dot-product-of-two-sparse-vectors/)  
**ML Context**: Large vocabulary bag-of-words, sparse feature embeddings, recommendation systems feature cross.

### Algorithmic Principle
Interviewers test representation choice and complexity trade-offs:
1. **Sorted Index-Value Pairs with Two Pointers**: $\mathcal{O}(L_1 + L_2)$ time, $\mathcal{O}(1)$ extra space, optimal cache locality.
2. **Binary Search Dispatch for Length Imbalance**: When $L_1 \ll L_2$, binary searching each index of the sparser vector into the denser vector achieves $\mathcal{O}(L_1 \log L_2)$.

- **Time Complexity**: $\min(L_1 + L_2, \; L_1 \log L_2)$ where $L = $ number of non-zero elements.
- **Space Complexity**: $\mathcal{O}(L)$ for pair storage.
- **Interview Trap**: Using Hash Maps, which have higher memory overhead and hash collision variance vs cache-friendly sorted lists.


In [ ]:
import bisect
from typing import List, Tuple

class SparseVector:
    """
    Stores non-zero values as sorted (index, value) tuples.
    Supports Two Pointers O(L1 + L2) and Binary Search O(L1 log L2).
    """
    def __init__(self, nums: List[int]):
        self.pairs: List[Tuple[int, int]] = [
            (i, val) for i, val in enumerate(nums) if val != 0
        ]

    def dot_product(self, other: 'SparseVector') -> int:
        p1, p2 = self.pairs, other.pairs
        l1, l2 = len(p1), len(p2)
        if l1 == 0 or l2 == 0:
            return 0

        # Dispatch to Binary Search if one vector is much sparser (e.g. >8x difference)
        if l1 * 8 < l2:
            return self._dot_binary_search(p1, p2)
        elif l2 * 8 < l1:
            return self._dot_binary_search(p2, p1)

        # Method 1: Two Pointers O(L1 + L2)
        res = 0
        i = j = 0
        while i < l1 and j < l2:
            idx1, val1 = p1[i]
            idx2, val2 = p2[j]
            if idx1 == idx2:
                res += val1 * val2
                i += 1
                j += 1
            elif idx1 < idx2:
                i += 1
            else:
                j += 1
        return res

    def _dot_binary_search(self, small: List[Tuple[int, int]], large: List[Tuple[int, int]]) -> int:
        res = 0
        large_indices = [idx for idx, _ in large]
        for idx, val in small:
            pos = bisect.bisect_left(large_indices, idx)
            if pos < len(large) and large[pos][0] == idx:
                res += val * large[pos][1]
        return res

# Verification Test
v1 = SparseVector([1, 0, 0, 2, 3])
v2 = SparseVector([0, 3, 0, 4, 0])
res = v1.dot_product(v2)
print("Dot product (v1 @ v2):", res)
assert res == 8  # index 3: 2 * 4 = 8

# Imbalanced vectors test
v_sparse = SparseVector([0] * 50 + [4] + [0] * 949)
v_dense = SparseVector([1] * 1000)
assert v_sparse.dot_product(v_dense) == 4
print("Imbalanced dot product test passed.")


---
## 9. Top-K Cosine Similarity Retrieval
**LeetCode Analog**: [LC 215: Kth Largest Element](https://leetcode.com/problems/kth-largest-element-in-an-array/), [LC 347: Top K Frequent Elements](https://leetcode.com/problems/top-k-frequent-elements/)  
**ML Context**: Vector search, candidate generation in recommendation systems, RAG nearest-neighbor retrieval.

### Algorithmic Principle
1. Compute cosine similarity: $\text{sim}(u, v) = \frac{u \cdot v}{\|u\|_2 \|v\|_2}$.
2. Maintain a **Min-Heap** (`heapq`) of fixed size $k$ containing `(similarity, doc_id)`.
3. For each candidate vector:
   - If heap has $< k$ elements, push candidate.
   - If `similarity > heap[0].similarity`, replace top via `heapreplace`.
4. Return candidates sorted descending by score.

- **Time Complexity**: $\mathcal{O}(N \cdot D + N \log k)$ where $N=$ candidates, $D=$ dimension.
- **Space Complexity**: $\mathcal{O}(k)$ auxiliary heap memory.
- **Interview Trap**: Full sorting $\mathcal{O}(N \log N)$ instead of heap $\mathcal{O}(N \log k)$; division by zero on zero-norm vectors.


In [ ]:
import math
import heapq
from typing import List, Tuple

def top_k_cosine_similarity(query: List[float], candidates: List[Tuple[str, List[float]]], k: int) -> List[Tuple[str, float]]:
    """
    Retrieves top-k candidates with highest cosine similarity using a min-heap.
    Time: O(N * D + N log k), Space: O(k).
    """
    def norm(vec: List[float]) -> float:
        return math.sqrt(sum(x * x for x in vec))

    q_norm = norm(query)
    if q_norm == 0.0 or k <= 0:
        return []

    min_heap: List[Tuple[float, str]] = []

    for doc_id, vec in candidates:
        v_norm = norm(vec)
        sim = sum(q * v for q, v in zip(query, vec)) / (q_norm * v_norm) if v_norm > 0 else 0.0

        if len(min_heap) < k:
            heapq.heappush(min_heap, (sim, doc_id))
        elif sim > min_heap[0][0]:
            heapq.heapreplace(min_heap, (sim, doc_id))

    # Sort descending
    return sorted([(doc_id, sim) for sim, doc_id in min_heap], key=lambda x: x[1], reverse=True)

# Verification Test
query = [1.0, 1.0, 0.0]
docs = [
    ("doc_1", [1.0, 1.0, 0.0]),   # Perfect alignment: sim = 1.0
    ("doc_2", [1.0, 0.0, 0.0]),   # Partial overlap: sim = 1 / sqrt(2) ~ 0.707
    ("doc_3", [0.0, 0.0, 5.0]),   # Orthogonal: sim = 0.0
    ("doc_4", [2.0, 2.0, 0.0])    # Scaled version: sim = 1.0
]

top2 = top_k_cosine_similarity(query, docs, k=2)
print("Top-2 retrieved documents:")
for doc_id, sim in top2:
    print(f"  {doc_id}: cosine similarity = {sim:.4f}")
assert len(top2) == 2 and abs(top2[0][1] - 1.0) < 1e-5


---
## 10. TF-IDF & Inverted Index Document Retrieval
**ML Context**: Lexical retrieval baseline, BM25 foundation, sparse keyword search in hybrid search engines.

### Algorithmic Principle
1. **Inverted Index**: Map each term $t \to \{ \text{doc\_id}: \text{count} \}$.
2. **Term Frequency**: $\text{TF}(t, d) = \frac{\text{count}(t, d)}{\text{total\_words}(d)}$.
3. **Inverse Document Frequency**: $\text{IDF}(t) = \log\left(\frac{N + 1}{\text{DF}(t) + 1}\right) + 1.0$ (smoothed).
4. Score document $d$ for query $Q$: $\text{Score}(d, Q) = \sum_{t \in Q} \text{TF}(t, d) \cdot \text{IDF}(t)$.

- **Time Complexity**: Indexing: $\mathcal{O}(\text{tokens})$; Querying: $\mathcal{O}(\sum_{t \in Q} \text{postings}(t))$.
- **Space Complexity**: $\mathcal{O}(\text{vocabulary} + \text{postings})$.
- **Interview Trap**: Not smoothing IDF (leading to division by zero on unseen query terms).


In [ ]:
import math
from collections import defaultdict, Counter
from typing import List, Tuple, Dict

class SimpleTFIDFIndex:
    """
    Inverted Index scoring documents with smoothed TF-IDF.
    """
    def __init__(self):
        # term -> dict(doc_id -> term_count)
        self.inverted_index: Dict[str, Dict[str, int]] = defaultdict(dict)
        self.doc_lengths: Dict[str, int] = {}
        self.total_docs = 0

    def add_document(self, doc_id: str, text: str):
        tokens = text.lower().split()
        self.doc_lengths[doc_id] = len(tokens)
        counts = Counter(tokens)
        for term, cnt in counts.items():
            self.inverted_index[term][doc_id] = cnt
        self.total_docs += 1

    def query(self, query_text: str, top_k: int = 5) -> List[Tuple[str, float]]:
        query_terms = query_text.lower().split()
        scores: Dict[str, float] = defaultdict(float)

        for term in query_terms:
            if term not in self.inverted_index:
                continue
            postings = self.inverted_index[term]
            df = len(postings)
            idf = math.log((self.total_docs + 1) / (df + 1)) + 1.0

            for doc_id, cnt in postings.items():
                tf = cnt / self.doc_lengths[doc_id]
                scores[doc_id] += tf * idf

        ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
        return ranked[:top_k]

# Verification Test
corpus = {
    "nlp_doc": "transformer neural networks natural language processing attention",
    "cv_doc": "convolutional neural networks computer vision image classification",
    "food_doc": "delicious italian pasta recipe dinner food"
}

index = SimpleTFIDFIndex()
for doc_id, text in corpus.items():
    index.add_document(doc_id, text)

query_res = index.query("neural networks vision", top_k=2)
print("Query results for 'neural networks vision':")
for doc_id, score in query_res:
    print(f"  {doc_id}: score = {score:.4f}")
assert query_res[0][0] == "cv_doc"  # Contains vision + neural + networks


---
## 11. Streaming Running Median (Two Heaps)
**LeetCode Analog**: [LC 295: Find Median from Data Stream](https://leetcode.com/problems/find-median-from-data-stream/)  
**ML Context**: Real-time feature drift detection, tracking online loss/latency distribution percentiles.

### Algorithmic Principle
Maintain two balanced heaps:
- `small`: **Max-Heap** (stores negated numbers) containing the lower half of data.
- `large`: **Min-Heap** containing the upper half of data.
- **Invariants**:
  1. Every element in `small` $\le$ every element in `large`.
  2. Size invariant: $\text{len}(\text{small}) \in \{ \text{len}(\text{large}), \text{len}(\text{large}) + 1 \}$.
- **Median**: If odd count, return top of `small`. If even, return average of tops.

- **Time Complexity**: $\mathcal{O}(\log N)$ insertion, $\mathcal{O}(1)$ median query.
- **Space Complexity**: $\mathcal{O}(N)$ stream storage.
- **Interview Trap**: Python only has min-heaps (`heapq`), so you must negate numbers (`-x`) to implement a max-heap.


In [ ]:
import heapq
from typing import List

class StreamingMedian:
    """
    Computes median from data stream using dual heaps.
    Time: O(log N) add, O(1) median. Space: O(N).
    """
    def __init__(self):
        self.small: List[float] = []  # Max-heap (stores -val)
        self.large: List[float] = []  # Min-heap (stores val)

    def add_num(self, num: float):
        # 1. Push to small (max-heap)
        heapq.heappush(self.small, -num)
        
        # 2. Ensure order invariant: max(small) <= min(large)
        if self.small and self.large and (-self.small[0] > self.large[0]):
            val = -heapq.heappop(self.small)
            heapq.heappush(self.large, val)

        # 3. Balance size invariant: len(small) == len(large) or len(small) == len(large) + 1
        if len(self.small) > len(self.large) + 1:
            val = -heapq.heappop(self.small)
            heapq.heappush(self.large, val)
        elif len(self.large) > len(self.small):
            val = heapq.heappop(self.large)
            heapq.heappush(self.small, -val)

    def find_median(self) -> float:
        if len(self.small) > len(self.large):
            return float(-self.small[0])
        return (-self.small[0] + self.large[0]) / 2.0

# Verification Test
sm = StreamingMedian()
stream = [5, 2, 8, 1, 9, 3]
medians = []
for x in stream:
    sm.add_num(x)
    medians.append(sm.find_median())

print("Stream numbers added:", stream)
print("Running medians:", medians)
assert medians[-1] == 4.0  # sorted: [1, 2, 3, 5, 8, 9] -> (3 + 5)/2 = 4.0


---
## 12. Edit Distance & Word Error Rate (WER)
**LeetCode Analog**: [LC 72: Edit Distance](https://leetcode.com/problems/edit-distance/)  
**ML Context**: Evaluation metric for Automatic Speech Recognition (ASR), OCR, and text generation. $\text{WER} = \frac{S + D + I}{N_{\text{ref}}}$.

### Algorithmic Principle
Dynamic Programming over token sequences of reference $R$ ($M$ words) and hypothesis $H$ ($N$ words):
$$DP[i][j] = \begin{cases} DP[i-1][j-1] & \text{if } R[i-1] = H[j-1] \\ 1 + \min(DP[i-1][j-1], DP[i-1][j], DP[i][j-1]) & \text{otherwise} \end{cases}$$
Where transitions correspond to **Substitution**, **Deletion**, and **Insertion**. Optimized to **1D DP row** to save memory.

- **Time Complexity**: $\mathcal{O}(M \cdot N)$.
- **Space Complexity**: $\mathcal{O}(\min(M, N))$ with row optimization.
- **Interview Trap**: Operating on characters instead of words/tokens (ASR WER operates on whitespace-delimited word tokens).


In [ ]:
def compute_wer(reference: str, hypothesis: str) -> float:
    """
    Computes Word Error Rate (WER) using Levenshtein distance on words.
    Time: O(M * N), Space: O(N).
    """
    ref = reference.strip().split()
    hyp = hypothesis.strip().split()
    m, n = len(ref), len(hyp)
    if m == 0:
        return float(n)

    # 1D DP row: dp[j] stores distance between ref[:i] and hyp[:j]
    dp = list(range(n + 1))

    for i in range(1, m + 1):
        prev = dp[0]
        dp[0] = i
        for j in range(1, n + 1):
            temp = dp[j]
            if ref[i - 1] == hyp[j - 1]:
                dp[j] = prev
            else:
                dp[j] = 1 + min(prev,      # substitution
                                dp[j],     # deletion
                                dp[j - 1]) # insertion
            prev = temp

    total_edits = dp[n]
    return total_edits / m

# Verification Test
ref_text = "the quick brown fox jumps over the lazy dog"
hyp_text = "the fast brown fox jumps above the dog"

wer = compute_wer(ref_text, hyp_text)
print(f"Reference:  '{ref_text}' ({len(ref_text.split())} words)")
print(f"Hypothesis: '{hyp_text}' ({len(hyp_text.split())} words)")
print(f"WER: {wer:.4f} (3 edits: quick->fast, over->above, lazy deleted)")
assert abs(wer - (3 / 9)) < 1e-5


---
## 13. Image Rotation by Arbitrary Angle
**ML / CV Context**: Data augmentation for computer vision, spatial transformers, affine matrix transformations without OpenCV/PIL.

### Algorithmic Principle
Why standard forward mapping fails:
- **Forward Mapping ($x, y \to x', y'$)** leaves blank pixels / holes because discrete integers do not map 1-to-1 onto the output grid.
- **Inverse Mapping ($x', y' \to x, y$)**: Iterate through every destination pixel $(x', y')$ and sample backward coordinates $(x, y) = R(-\theta)(x' - c_x, y' - c_y) + (c_x, c_y)$.
- **Bilinear Interpolation**: For non-integer source coordinates $(x, y)$, sample the 4 neighboring integer pixels using fractional distances $dx, dy$:
  $$I(x, y) = (1 - dx)(1 - dy) I_{00} + dx(1 - dy) I_{10} + (1 - dx)dy I_{01} + dx \cdot dy I_{11}$$

- **Time Complexity**: $\mathcal{O}(H \cdot W)$.
- **Space Complexity**: $\mathcal{O}(H \cdot W)$ for output image.
- **Interview Trap**: Using forward mapping instead of inverse mapping; out-of-bounds index errors when $(x_1, y_1)$ touches image boundary.


In [ ]:
import math
from typing import List

def rotate_image(image: List[List[float]], angle_degrees: float, fill_value: float = 0.0) -> List[List[float]]:
    """
    Rotates a 2D grayscale image by an arbitrary angle clockwise around its center.
    Uses Inverse Mapping + Bilinear Interpolation.
    Time: O(H * W), Space: O(H * W).
    """
    H = len(image)
    W = len(image[0]) if H > 0 else 0
    if H == 0 or W == 0:
        return []

    cx = (W - 1) / 2.0
    cy = (H - 1) / 2.0

    # Clockwise rotation in image coordinates (y points down)
    theta = math.radians(angle_degrees)
    cos_t = math.cos(theta)
    sin_t = math.sin(theta)

    out = [[fill_value for _ in range(W)] for _ in range(H)]

    for yp in range(H):
        for xp in range(W):
            # Inverse mapping: rotate backwards by -theta
            dx_dest = xp - cx
            dy_dest = yp - cy
            x_src = dx_dest * cos_t + dy_dest * sin_t + cx
            y_src = -dx_dest * sin_t + dy_dest * cos_t + cy

            # Bilinear interpolation bounds
            x0 = math.floor(x_src)
            x1 = x0 + 1
            y0 = math.floor(y_src)
            y1 = y0 + 1

            if 0 <= x0 < W and 0 <= x1 < W and 0 <= y0 < H and 0 <= y1 < H:
                wx = x_src - x0
                wy = y_src - y0

                val = ( (1 - wx) * (1 - wy) * image[y0][x0] +
                        wx * (1 - wy) * image[y0][x1] +
                        (1 - wx) * wy * image[y1][x0] +
                        wx * wy * image[y1][x1] )
                out[yp][xp] = round(val, 2)
            elif 0 <= round(x_src) < W and 0 <= round(y_src) < H:
                out[yp][xp] = image[round(y_src)][round(x_src)]

    return out

# Verification Test: Rotate 5x5 pattern with cross
img = [
    [0.0, 0.0, 1.0, 0.0, 0.0],
    [0.0, 0.0, 1.0, 0.0, 0.0],
    [1.0, 1.0, 1.0, 1.0, 1.0],
    [0.0, 0.0, 1.0, 0.0, 0.0],
    [0.0, 0.0, 1.0, 0.0, 0.0]
]

print("Original 5x5 pattern:")
for row in img:
    print(" ", [int(v) for v in row])

rotated_45 = rotate_image(img, 45.0)
print("\nRotated by 45 degrees (bilinear interpolation):")
for row in rotated_45:
    print(" ", [round(v, 2) for v in row])

assert rotate_image(img, 0.0) == img
assert rotated_45[2][2] == 1.0  # Center stays invariant
print("\nImage rotation test passed!")


---
## 🎯 MLE Algorithmic Interview Quick Reference

| Pattern | Problem | Key Library / Technique | Complexity |
|---|---|---|---|
| **Binary Search** | Weighted Sampling | `bisect.bisect_right` on Prefix Sums | $\mathcal{O}(\log N)$ query |
| **Streaming** | Reservoir Sampling | Replace element with prob $k/(i+1)$ | $\mathcal{O}(N)$ time, $\mathcal{O}(k)$ space |
| **Two Pointers** | Sort Transformed Array | Monotonicity from ends towards vertex | $\mathcal{O}(N)$ time |
| **Divide & Conquer** | Ternary Search | Discard $1/3$ of interval per iteration | $\mathcal{O}(\log(1/\epsilon))$ time |
| **Binning** | Expected Calibration Error | Equal-width bucket accumulation | $\mathcal{O}(N + M)$ time |
| **Sorting / Rank Sum**| Exact ROC-AUC | Wilcoxon-Mann-Whitney with tied ranks | $\mathcal{O}(N \log N)$ time |
| **Sweep-Line** | Average Precision (PR-AUC) | Descending score threshold integration | $\mathcal{O}(N \log N)$ time |
| **Two Pointers / Bisect**| Sparse Vector Dot Product | Sorted `(index, value)` pairs | $\mathcal{O}(L_1 + L_2)$ or $\mathcal{O}(L_1 \log L_2)$ |
| **Min-Heap** | Top-K Cosine Retrieval | `heapq.heapreplace` of size $k$ | $\mathcal{O}(N \cdot D + N \log k)$ |
| **Hash Map / Postings**| TF-IDF Inverted Index | Term frequency $\times$ smoothed IDF | $\mathcal{O}(\text{postings})$ |
| **Two Heaps** | Streaming Median | Max-heap (lower) + Min-heap (upper) | $\mathcal{O}(\log N)$ add, $\mathcal{O}(1)$ median |
| **Dynamic Programming**| Edit Distance / WER | 1D space-optimized DP table | $\mathcal{O}(M \cdot N)$ time, $\mathcal{O}(N)$ space |
| **Coordinate Geometry** | Arbitrary Image Rotation | Inverse mapping + Bilinear interpolation | $\mathcal{O}(H \cdot W)$ |
